# Module 2: Using Existing Environments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/openenv-course/blob/main/module-2/notebook.ipynb)

In this notebook, you'll:
1. Write 4 different policies for the Catch game
2. Compare their performance
3. Run a policy competition
4. Switch to a different game with the same client code

## Setup

In [ ]:
!pip install openenv-core -q

import os
import sys
import random
from typing import List

if not os.path.exists('OpenEnv'):
    !git clone https://github.com/meta-pytorch/OpenEnv.git

repo = os.path.abspath('OpenEnv')
sys.path.insert(0, repo)
sys.path.insert(0, os.path.join(repo, 'src'))

In [ ]:
from envs.openspiel_env import OpenSpielEnv
from envs.openspiel_env.models import OpenSpielAction, OpenSpielObservation

## Understanding the Catch Game

Catch is a simple game:
- A ball falls from the top of the screen
- You control a paddle at the bottom
- Goal: move the paddle to catch the ball

Actions:
- 0 = LEFT
- 1 = STAY
- 2 = RIGHT

The observation contains:
- `info_state`: A flattened vector representing the game state
- `legal_actions`: Which actions are valid
- `reward`: +1 for catching, 0 otherwise
- `done`: Whether the episode is over

## Helper Functions

In [ ]:
def find_ball(info_state: List[float], rows: int = 10, cols: int = 5) -> int:
    """Find the column position of the ball."""
    for i, val in enumerate(info_state):
        if val == 1.0:  # Ball is represented as 1.0
            return i % cols
    return cols // 2  # Default to middle if ball not found

def find_paddle(info_state: List[float], rows: int = 10, cols: int = 5) -> int:
    """Find the column position of the paddle."""
    # Paddle is in the last row
    last_row_start = (rows - 1) * cols
    for i in range(cols):
        if info_state[last_row_start + i] == 2.0:  # Paddle is represented as 2.0
            return i
    return cols // 2  # Default to middle if paddle not found

def run_episode(env, policy, policy_name: str, max_steps: int = 100) -> dict:
    """Run a single episode with a given policy."""
    result = env.reset()
    total_reward = 0
    steps = 0
    
    while not result.observation.done and steps < max_steps:
        action_id = policy(result.observation, steps)
        action = OpenSpielAction(action_id=action_id, game_name="catch")
        result = env.step(action)
        total_reward += result.observation.reward or 0
        steps += 1
    
    return {
        "policy": policy_name,
        "reward": total_reward,
        "steps": steps,
        "success": total_reward > 0
    }

## Policy 1: Random

Baseline policy - just pick a random action.

In [ ]:
def random_policy(obs: OpenSpielObservation, step: int) -> int:
    """Choose a random legal action."""
    return random.choice(obs.legal_actions)

## Policy 2: Always Stay

A terrible policy - never move the paddle.

In [ ]:
def stay_policy(obs: OpenSpielObservation, step: int) -> int:
    """Always stay in place."""
    return 1  # STAY

## Policy 3: Smart Heuristic

An optimal policy - move towards the ball.

In [ ]:
def smart_policy(obs: OpenSpielObservation, step: int) -> int:
    """Move paddle towards the ball."""
    ball_col = find_ball(obs.info_state)
    paddle_col = find_paddle(obs.info_state)
    
    if paddle_col < ball_col:
        return 2  # RIGHT
    elif paddle_col > ball_col:
        return 0  # LEFT
    else:
        return 1  # STAY

## Policy 4: Epsilon-Greedy

Explore randomly at first, then exploit the smart policy.

In [ ]:
def epsilon_greedy_policy(obs: OpenSpielObservation, step: int) -> int:
    """Epsilon-greedy: explore with decreasing probability."""
    epsilon = max(0.1, 1.0 - step / 100)
    
    if random.random() < epsilon:
        return random.choice(obs.legal_actions)
    else:
        return smart_policy(obs, step)

## Run Policy Competition

Let's test all policies and compare their performance.

In [ ]:
policies = [
    (random_policy, "Random"),
    (stay_policy, "Always Stay"),
    (smart_policy, "Smart Heuristic"),
    (epsilon_greedy_policy, "Epsilon-Greedy")
]

num_episodes = 10
results = []

print("Running policy competition...\n")

with OpenSpielEnv(base_url="https://openenv-openspiel-catch.hf.space").sync() as env:
    for policy, name in policies:
        print(f"Testing {name} policy...")
        policy_results = []
        
        for episode in range(num_episodes):
            result = run_episode(env, policy, name)
            policy_results.append(result)
        
        # Calculate statistics
        avg_reward = sum(r["reward"] for r in policy_results) / num_episodes
        success_rate = sum(r["success"] for r in policy_results) / num_episodes * 100
        avg_steps = sum(r["steps"] for r in policy_results) / num_episodes
        
        results.append({
            "Policy": name,
            "Avg Reward": f"{avg_reward:.2f}",
            "Success Rate": f"{success_rate:.0f}%",
            "Avg Steps": f"{avg_steps:.1f}"
        })
        
        print(f"  Avg Reward: {avg_reward:.2f}")
        print(f"  Success Rate: {success_rate:.0f}%")
        print(f"  Avg Steps: {avg_steps:.1f}\n")

## Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print("\n" + "="*60)
print("POLICY COMPETITION RESULTS")
print("="*60)
print(df.to_string(index=False))
print("="*60)

## Switch to a Different Game

Let's try the same smart policy on Tic-Tac-Toe. Notice how the client code stays the same!

In [ ]:
print("Switching to Tic-Tac-Toe...\n")

with OpenSpielEnv(base_url="https://openenv-openspiel-tictactoe.hf.space").sync() as env:
    result = env.reset()
    print(f"Game started! Legal actions: {result.observation.legal_actions}")
    
    # Take a random action (center position)
    action = OpenSpielAction(action_id=4, game_name="tic_tac_toe")
    result = env.step(action)
    
    print(f"\nAfter first move:")
    print(f"  Current player: {result.observation.current_player_id}")
    print(f"  Legal actions: {result.observation.legal_actions}")
    print(f"  Done: {result.observation.done}")

## Key Takeaways

1. **Same interface, different games**: The client code is identical across games
2. **Type-safe observations**: Your IDE knows what fields are available
3. **Policy flexibility**: You can easily compare different strategies
4. **Reusable code**: The `run_episode` function works with any policy

In Module 3, you'll learn how to deploy your own environments to make them available to others.

## Exercise: Create Your Own Policy

Try creating a new policy that:
1. Moves more aggressively (two steps at a time if possible)
2. Uses a different exploration strategy
3. Adapts based on previous successes/failures

In [ ]:
# Your code here
def my_custom_policy(obs: OpenSpielObservation, step: int) -> int:
    # TODO: Implement your policy
    pass
